# Figure 3

Merged source notebooks. Run top to bottom. Historical methods, inactive blocks and fixed plot values are preserved. Part B reloads the working data and model. Inputs are read-only. Figure exports use figure3_outputs.


In [ ]:
from pathlib import Path
FIGURE3_OUTPUT_DIR = Path.cwd() / "figure3_outputs"
FIGURE3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## A. Historical prediction contrasts and SHAP (training through 2010)


In [ ]:
# Data preparation.
import numpy as np 
import pandas as pd 
from sklearn.preprocessing import StandardScaler
df = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119.xlsx") 
label = ['Cross-Border Basin Agreements','Collaborative River Basin Planning','Joint Water Allocation Models',
         'Joint Data-Sharing Systems','Transboundary Water Quality Standards','Coordinated Hydropower Operations'] 
df.loc[df['Both'] == 1, ['Conflict', 'Cooperation']] = 1  
df['Cooperation'] = df[label].any(axis=1).astype(int)
df['potential_Conflict'] = df['Conflict'] 
SEdata = ['gdp_percapita', 'export', 'population_density', 'rural_population', 'precipitation', 'bws_score']
df = df[['year', 'ccode_iso3', 'Country_count', 'Conflict', 'potential_Conflict'] + SEdata + label]
pd.set_option('future.no_silent_downcasting', True)  
df = df.replace('..', np.nan)  
scaler = StandardScaler()  
df[SEdata] = scaler.fit_transform(df[SEdata])  
df = df.dropna(subset=df.columns.difference(['Country_count']))
num_rows = df.shape[0]  
print("Number of rows:", num_rows)  


In [ ]:
# Optional mode co-occurrence (inactive).
'''
df_label = df[label]
labels = df_label.columns.tolist()
sums = {}
for label in labels:
    other_labels = [l for l in labels if l != label]
    sums[label] = df_label[df_label[label] == 1][other_labels].sum(axis=1).sum()
for label, sum_ in sums.items():
    print(f"When '{label}' equals 1, sum of other indicators: {sum_}")
'''


In [ ]:
# Train through 2010; test later conflict-only rows.
y = 'Country_count'
x = df[['Conflict'] + SEdata + label] 
mask_train = df['year'] <= 2010  
x_train = x[mask_train]  
y_train = df[y][mask_train]
#mask_test = df['year'] > 2010
#x_test = x[mask_test]  
#y_test = df[y][mask_test]
mask_year = df['year'] > 2010  
mask_conflict = df['potential_Conflict'] == 1   
mask_test = mask_year & mask_conflict  
x_test = x[mask_test]  
y_test = df[y][mask_test]


In [ ]:
# Prediction contrasts; Conflict changes along with mode indicators.
import pandas as pd  
from sklearn.ensemble import RandomForestRegressor  
rfr = RandomForestRegressor(random_state=42)  
rfr.fit(x_train, y_train)  
x_test_case1 = x_test.copy()  
x_test_case1[label] = 0  
y_pred_case1 = rfr.predict(x_test_case1)  
df_copy = df.copy()  
df_copy.loc[mask_test, 'Noncooperation'] = y_pred_case1
x_test_case2 = x_test.copy()  
x_test_case2['Conflict'] = 0  
y_pred_case2 = rfr.predict(x_test_case2)  
df_copy.loc[mask_test, 'Cooperation'] = y_pred_case2  
df_copy.loc[mask_test, 'Combined Mitigation'] = df_copy.loc[mask_test, 'Cooperation'] - df_copy.loc[mask_test, 'Noncooperation']  
print(df_copy[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Combined Mitigation']])
#df_copy[mask_test][['year', 'ccode_iso3', 'Noncooperation', 'Cooperation', 'Combined Mitigation'] + label].to_excel('C:\\Users\\A\\Desktop\\Combined Mitigation.xlsx', index=False)


In [ ]:
# Optional importance plot (inactive).
'''
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
importances = rfr.feature_importances_
feature_importances = dict(zip(label, importances))   
print("Feature names and importance:")  
for feature, importance in feature_importances.items():  
    print(f"{feature}: {importance:.4f}")
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)
sedata_length = len(SEdata)
label_indices = np.arange(sedata_length + 1, sedata_length + 1 + len(label))
label_importances = importances[label_indices]
label_importances_std = std[label_indices]
sorted_indices = np.argsort(label_importances)[::-1]  
sorted_label_importances = label_importances[sorted_indices]  
sorted_label_importances_std = label_importances_std[sorted_indices]  
sorted_labels = [label[i] for i in sorted_indices]  
color = "#D20419"
plt.figure(figsize=(4, 4.5))  
plt.title("Feature Importances")  
plt.bar(range(len(sorted_labels)), sorted_label_importances, color=color, align="center", 
        yerr=sorted_label_importances_std, capsize=5, width=0.8)
plt.xticks(range(len(sorted_labels)), sorted_labels, rotation=30)  
plt.xlim([-1, len(sorted_labels)])  
plt.tight_layout()  
#plt.savefig("C:\\Users\\A\\Desktop\\Feature Importances.svg", format='svg')
'''


In [ ]:
# Optional RFE/MAE selection (inactive).
'''
import numpy as np  
import matplotlib.pyplot as plt  
from sklearn.feature_selection import RFE  
from sklearn.ensemble import RandomForestRegressor  
from sklearn.metrics import mean_absolute_error  
best_mae_features = None    
best_mae = float('inf')    
mae_results = []    
for n_features in range(8, len(x.columns) + 1):
    rfe = RFE(estimator=RandomForestRegressor(random_state=42), n_features_to_select=n_features, step=1)    
    rfe.fit(x_train, y_train.values.ravel())    
    selected_features_indices = rfe.support_    
    selected_features = [x.columns[i] for i, selected in enumerate(selected_features_indices) if selected]    
    rfr = RandomForestRegressor(random_state=42)    
    rfr.fit(x_train[selected_features], y_train.values.ravel())    
    y_pred = rfr.predict(x_test[selected_features])    
    mae = mean_absolute_error(y_test, y_pred)    
    mae_results.append((selected_features, mae))    
    if mae < best_mae:    
        best_mae = mae    
        best_mae_features = selected_features    
    print(f"Number of features: {n_features}; Feature subset: {', '.join(selected_features)}. MAE: {mae:.4f}")    
print(f"Best MAE and feature subset: MAE: {best_mae:.4f}; Features: {', '.join(best_mae_features)}")    

plt.figure(figsize=(10, 5))
plt.bar(range(len(mae_results)), [mae for _, mae in mae_results], color='blue')  
for i, (features, mae) in enumerate(mae_results):  
    plt.text(i, mae + 0.01 * (max([mae for _, mae in mae_results]) - min([mae for _, mae in mae_results])),  
             f'{mae:.4f}', ha='center', va='bottom', fontsize=8)  
plt.xlabel('Number of Features')  
plt.ylabel('MAE')  
plt.title('MAE for Different Number of Features')  
plt.xticks(range(len(mae_results)), [i + 1 for i in range(len(mae_results))]) 
y_values = [mae for _, mae in mae_results]  
plt.ylim(bottom=min(y_values) - 0.1 * (max(y_values) - min(y_values)), top=max(y_values) + 0.05 * (max(y_values) - min(y_values)))  
plt.show()
'''


In [ ]:
# Optional combined contrast and t interval (inactive).
'''
import pandas as pd  
from scipy import stats 
IntegrativeGovernance = ['Cross-Border Basin Agreements', 'Collaborative River Basin Planning', 'Joint Water Allocation Models']
Non_IG = [lbl for lbl in label if lbl not in IntegrativeGovernance]
x_test_case3_1 = x_test.copy()  
x_test_case3_1['Conflict'] = 0  
#x_test_case3_1 = x_test_case3_1[(x_test_case3_1[IntegrativeGovernance] == 1).all(axis=1) & (x_test_case3_1[Non_IG] == 0).all(axis=1)]
x_test_case3_1 = x_test_case3_1[(x_test_case3_1[IntegrativeGovernance] == 1).all(axis=1)]
x_test_case3_1[IntegrativeGovernance] = 1
x_test_case3_1[Non_IG] = 0
x_test_case3_2 = x_test.copy()  
x_test_case3_2 = x_test_case3_2[(x_test_case3_2[IntegrativeGovernance] == 1).all(axis=1) & (x_test_case3_1[Non_IG] == 0).all(axis=1)]
#x_test_case3_2 = x_test_case3_2[(x_test_case3_2[IntegrativeGovernance] == 1).all(axis=1) & (x_test_case3_2[Non_IG] == 0).all(axis=1)]
x_test_case3_2 = x_test_case3_2[(x_test_case3_2[IntegrativeGovernance] == 1).all(axis=1)]
x_test_case3_2[label] = 0

# Cooperation
#x_test_case3_1 = x_test.copy()
# Noncooperation
#x_test_case3_2 = x_test.copy()

y_pred_case3_1 = rfr.predict(x_test_case3_1)
y_pred_case3_2 = rfr.predict(x_test_case3_2)
differences = y_pred_case3_1 - y_pred_case3_2
mean_difference = np.mean(differences) 

std_error = np.std(differences, ddof=1) / np.sqrt(len(differences)) 
t_critical = stats.t.ppf(0.975, df=len(differences)-1)
ci_95_lower = mean_difference - t_critical * std_error
ci_95_upper = mean_difference + t_critical * std_error

print(f"Mean difference: {mean_difference}")  
print(f"95% confidence interval: [{ci_95_lower}, {ci_95_upper}]")  
'''


In [ ]:
# Forest plot with fixed estimates.
import matplotlib.pyplot as plt

effect = -1.11
ci_low = -1.90
ci_high = -0.32

fig, ax = plt.subplots(figsize=(2.5, 1))

ax.axvline(x=0, color='#888888', linestyle='--', linewidth=1.4)

ax.errorbar(
    x=effect,
    y=0,
    xerr=[[effect - ci_low],
          [ci_high - effect]],
    fmt='s',
    markersize=7,
    color='#115757',
    ecolor='#115757',
    elinewidth=1.5,
    capsize=4,
    capthick=1.5
)

ax.set_yticks([])
#ax.set_xlabel("Mitigation Effect")

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

#ax.text(effect, 0.15, "-1.11", ha='center', fontsize=10)
#ax.text((ci_low + ci_high)/2, -0.2, "95% CI [-1.90, -0.32]", ha='center', fontsize=9)

ax.set_xlim(-2.1, 0.1)
ax.set_xticks([-2.0, -1.5, -1.0,-0.5, 0.0])
ax.spines['bottom'].set_linewidth(1.0)
ax.spines['bottom'].set_color('#000000')
ax.set_xticklabels([])

ax.set_ylim(-0.5, 0.5)

plt.tight_layout()

#plt.savefig(str(FIGURE3_OUTPUT_DIR / 'historical_forest_plot.png'), dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
# Single-mode prediction contrasts.
results_mean = {}  
results_diff = {}  
for l in label:  
    x_test_case4 = x_test.copy()  
    x_test_case4['Conflict'] = 0  
    x_test_case4[label] = 0  
    x_test_case4[l] = 1  
    y_pred_case = rfr.predict(x_test_case4)  
    diff = y_pred_case - y_pred_case1    
    results_diff[l] = diff.tolist()  
    diff_mean = np.mean(diff)  
    results_mean[l] = diff_mean
mask_test = mask_test.reindex(df.index, fill_value=False)  
for l, diff_list in results_diff.items():  
    df[f'Cooperation_Diff_{l}'] = np.nan   
    df.loc[mask_test, f'Cooperation_Diff_{l}'] = diff_list   
df_mask_test = df[mask_test]  
#print(df_mask_test[['year', 'ccode_iso3'] + [col for col in df_mask_test.columns if 'Cooperation_Diff' in col]]) 
#df_mask_test.to_excel('C:\\Users\\A\\Desktop\\results_diff.xlsx', index=False)

results_mean_df = pd.DataFrame(list(results_mean.items()), columns=['Variable', 'Mitigating Effect'])  
print(results_mean_df)  
# results_diffmean_df.to_excel('C:\\Users\\A\\Desktop\\results_diffmean.xlsx', index=False)


In [ ]:
# Row-based t intervals.
import numpy as np  
import pandas as pd  
from scipy import stats  
results_ci = {}  
for l in label:  
    diff = np.array(results_diff[l])
    diff_mean = results_mean[l]
    std_err = np.std(diff, ddof=1) / np.sqrt(len(diff))
    h = stats.t.ppf((1 + 0.95) / 2., len(diff) - 1)    
    ci_lower = diff_mean - h * std_err  
    ci_upper = diff_mean + h * std_err  
    results_ci[l] = (ci_lower, ci_upper)  
results_ci_df = pd.DataFrame(list(results_ci.items()), columns=['Variable', '95% CI'])  
results_ci_df['Lower Bound'] = results_ci_df['95% CI'].apply(lambda x: x[0])  
results_ci_df['Upper Bound'] = results_ci_df['95% CI'].apply(lambda x: x[1])  
results_ci_df.drop(columns=['95% CI'], inplace=True)  
print(results_ci_df)  
# results_ci_df.to_excel('C:\\Users\\A\\Desktop\\results_ci.xlsx', index=False)


In [ ]:
# Plot fixed single-mode estimates.
import matplotlib.pyplot as plt  
import matplotlib as mpl  
mpl.rcParams['font.family'] = 'Arial'  
mpl.rcParams['font.size'] = 8  
scores = [  
-0.784894,   # Cross-Border Basin Agreements          
-0.867872,   # Collaborative River Basin Planning
-0.835957,   # Joint Water Allocation Models
-0.864894,   # Joint Data-Sharing Systems
-0.878936,   # Transboundary Water Quality Standards
-0.890319,   # Coordinated Hydropower Operations
] 
reference_values1 = [-0.965230, -1.082376, -1.056227, -1.078656, -1.099028, -1.109722]  
reference_values2 = [-0.604557, -0.653369, -0.615688, -0.651131, -0.658845, -0.670916]  
plt.figure(figsize=(3, 5))  
plt.plot(scores, range(len(scores)), 'o-', color='#007878', markersize=11, linewidth=1, label='Scores')  
plt.plot(reference_values1, range(len(reference_values1)), 'o-', color='#BDD7EE', markersize=9, linewidth=1, label='Reference 1')   # 5F93BB
plt.plot(reference_values2, range(len(reference_values2)), 'o-', color='#BDD7EE', markersize=9, linewidth=1, label='Reference 2')  
plt.gca().invert_yaxis()  
plt.xlim(max(scores + reference_values1 + reference_values2), min(scores + reference_values1 + reference_values2))  
plt.xlim(min(reference_values1) - 0.1, max(reference_values2) + 0.5) 
plt.yticks(range(len(scores)), label)  
#for i, score in enumerate(scores):  
#for i, value in enumerate(reference_values1):  
#    plt.text(value - 0.1, i, f'{value:.4f}', va='center', color='#5F93BB')  
#for i, value in enumerate(reference_values2):  
#    plt.text(value + 0.1, i, f'{value:.4f}', va='center', color='#9AC5E7')   
ax = plt.gca()  
for spine in ax.spines.values():  
    spine.set_visible(False)  
ax.spines['left'].set_visible(True)  
ax.set_xticks([])   
plt.savefig(
    str(FIGURE3_OUTPUT_DIR / 'historical_line_0.png'),
    dpi=300,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# SHAP summary.
import shap
import matplotlib
import matplotlib.pyplot as plt
### SHapley Additive exPlanations (SHAP)
explainer = shap.Explainer(rfr, x_train)  
shap_values = explainer(x_test)
x_display = x_test[label]  
shap.summary_plot(shap_values[:, x_test.columns.get_indexer(label)], x_display, show=False)
fig, ax = plt.gcf(), plt.gca()
ax.tick_params(axis='both', which='major', labelsize=8)
ax.set_facecolor('#FAFAFA')
#fig.savefig("C:\\Users\\A\\Desktop\\shap_summary.svg", format='svg')


In [ ]:
# Alternative SHAP summary.
import shap
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =========================
# =========================
explainer = shap.Explainer(rfr, x_train)
shap_values = explainer(x_test)

# =========================
# =========================
custom_cmap = LinearSegmentedColormap.from_list(
    "custom_shap",
    ["#F5DF8B", "#007878"]
)

# =========================
# =========================
x_display = x_test[label]

# =========================
# =========================
shap.summary_plot(
    shap_values[:, x_test.columns.get_indexer(label)],
    x_display,
    show=False,
    cmap=custom_cmap
)

# =========================
# =========================
fig, ax = plt.gcf(), plt.gca()

ax.tick_params(axis='both', which='major', labelsize=8)

ax.set_facecolor('#FAFAFA')

for col in ax.collections:
    col.set_sizes([50])
    col.set_edgecolor("#2b2b2b")
    col.set_linewidth(0.5)
    col.set_alpha(0.99)

ax.axvline(0, color='gray', linewidth=0.8)

# =========================
# =========================
fig.savefig(
    str(FIGURE3_OUTPUT_DIR / 'historical_shap_summary.png'),
    dpi=600,
    bbox_inches='tight'
)
plt.show()


In [ ]:
# Mean signed SHAP interactions.
import numpy as np  
from sklearn.ensemble import RandomForestRegressor  
import shap  
import pandas as pd  
rfr = RandomForestRegressor(random_state=42)  
rfr.fit(x_train, y_train)  
explainer = shap.TreeExplainer(rfr)  
shap_interaction_values = explainer.shap_interaction_values(x_test)  
feature_names = x_test.columns  
num_features = len(feature_names)
interaction_matrix = np.zeros((num_features, num_features))
for i in range(num_features):
    for j in range(i+1, num_features):
        average_interaction = np.mean(shap_interaction_values[:, i, j])
        interaction_matrix[i, j] = average_interaction
        interaction_matrix[j, i] = average_interaction
interaction_df = pd.DataFrame(interaction_matrix, index=feature_names, columns=feature_names)
#interaction_df.to_excel("C:\\Users\\A\\Desktop\\interaction_df.xlsx", index=False)
print(interaction_df)


In [ ]:
# Mode/context correlation heatmap.
import matplotlib.pyplot as plt  
import seaborn as sns  
import numpy as np  
import pandas as pd 
from matplotlib.colors import LinearSegmentedColormap
#correlation_matrix = df[SEdata + label].corr() 
correlation_matrix = df[df['year'] <= 2019][SEdata + label].corr()
correlation_matrix = pd.DataFrame(correlation_matrix, index=label, columns=SEdata)
#correlation_matrix.to_excel('C:\\Users\\A\\Desktop\\correlation_matrix.xlsx', index=True)
'''
custom_colors = [
    (0/255, 51/255, 102/255),    # #003366
    (33/255, 113/255, 181/255),  # #2171B5
    (66/255, 146/255, 198/255),  # #4292C6
    (107/255, 174/255, 214/255), # #6BAED6
    (158/255, 188/255, 218/255), # #9EBCDA
    (245/255, 245/255, 245/255), # #F5F5F5
    (248/255, 192/255, 164/255), # 
    (223/255, 116/255, 92/255),  # 
    (192/255, 52/255, 56/255),   # 
    (155/255, 16/255, 39/255),   # 
    (103/255, 0/255, 31/255)     # 
]
cmap = LinearSegmentedColormap.from_list('custom_cmap', custom_colors)
'''

# ==============================================================================
from matplotlib.colors import ListedColormap
discrete_hex_colors = [
    '#E3F8A6',
    '#BEE48C',
    '#9DD19E',
    '#5BB5B4',
    '#2773A7'
]

def hex_to_rgb_norm(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) / 255.0 for i in (0, 2, 4))

colors_normalized = [hex_to_rgb_norm(c) for c in discrete_hex_colors]

cmap = ListedColormap(colors_normalized) 
# ==============================================================================

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.figure(figsize=(6, 5))
ax = sns.heatmap(correlation_matrix, cmap=cmap, annot=False, fmt=".3f", xticklabels=True, yticklabels=True, square=True,
            cbar_kws={"shrink": .6}, linewidths=1, vmin=-0.25, vmax=0.1)
cbar = ax.collections[0].colorbar  
cbar.format = "%.3f"  
vmin = cbar.vmin  
vmax = cbar.vmax  
vmid = (vmin + vmax) / 2  
cbar.set_ticks([vmin, vmid, vmax])  

plt.xticks(rotation=30, ha='right')
plt.yticks(va='center')
plt.tight_layout()
#plt.savefig(
#    str(FIGURE3_OUTPUT_DIR / 'historical_correlation_matrix.png'),
#    bbox_inches='tight'
#)
#correlation_matrix.to_csv('C:\\Users\\A\\Desktop\\correlation_matrix.csv', index=True)


In [ ]:
# Mode/mode correlation heatmap.
correlation_matrix = df[label].corr() 
correlation_matrix = df[df['year'] <= 2019][label].corr()
mask = np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
plt.figure(figsize=(6, 5))
ax = sns.heatmap(correlation_matrix, cmap=cmap, annot=False, fmt=".3f", xticklabels=True, yticklabels=True, square=True,
            cbar_kws={"shrink": .6}, linewidths=1, vmin=0.1, vmax=1, mask=mask)
cbar = ax.collections[0].colorbar  
cbar.format = "%.3f"  
vmin = cbar.vmin  
vmax = cbar.vmax  
vmid = (vmin + vmax) / 2  
cbar.set_ticks([vmin, vmid, vmax])  
plt.xticks(rotation=30, ha='right')
plt.yticks(va='center')
plt.tight_layout()
#plt.savefig(
#    str(FIGURE3_OUTPUT_DIR / 'historical_correlation_matrix2.png'),
#    bbox_inches='tight'
#)


## B. Historical-model feature importance (training through 2019)


In [ ]:
# Read historical and SSP5 data.
import pandas as pd   
df1 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119.xlsx") 
df2 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\E_ssp_predication\ssp1-5_future_data.xlsx", sheet_name='ssp5')  
df2 = df2[df2['year'] == 2050]  
#df3 = pd.read_excel('C:\\Users\\A\\Desktop\\RF_Water\\D_strategy_onehot\\20240926_2050ssp5_strategy_onehot.xlsx')
label = ['Joint Data-Sharing Systems','Cross-Border Basin Agreements','Transboundary Water Quality Standards',
         'Coordinated Hydropower Operations','Joint Water Allocation Models','Collaborative River Basin Planning'] 
#df3_labels = df3[['country_code'] + label]  
#df2 = pd.merge(df2, df3_labels, on='country_code', how='left')  


In [ ]:
# Append 2050 inputs and assign risk flags.
import pandas as pd
df2.rename(columns={  
    'country': 'Country',
    'country_code': 'ccode_iso3',
    'population_density_ssp5': 'population_density', 
    'GDP per capita_ssp5(constant 2015 US$)': 'gdp_percapita',  
    'rural_population_ssp5(%)': 'rural_population',
    'precipitation_ssp5': 'precipitation',  
    'bws_ssp5': 'bws_score'}, inplace=True)  
df = pd.concat([df1, df2], ignore_index=True)  
df4 = pd.read_excel(r"E:\AExperiment\11Water\E_RF_Data\B_Clustered_panel\03_Clustered_panel_1119\Clustered_Conflict_Probability_241119.xlsx")  
high_conflict_country = df4[df4['Cluster'] == 1]  
high_conflict_country = high_conflict_country['ccode_iso3'].tolist()  
df.loc[(df['year'] == 2050) & (df['ccode_iso3'].isin(high_conflict_country)), 'Conflict'] = 1


In [ ]:
# Preprocessing and combined-data standardization.
import numpy as np  
from sklearn.preprocessing import StandardScaler
df.loc[df['Both'] == 1, ['Conflict', 'Cooperation']] = 1  
df['Cooperation'] = df[label].any(axis=1).astype(int)
SEdata = ['gdp_percapita', 'export', 'population_density', 'rural_population', 'precipitation', 'bws_score']
df = df[['year', 'ccode_iso3', 'Country_count', 'Conflict'] + SEdata + label]
pd.set_option('future.no_silent_downcasting', True)  
df = df.replace('..', np.nan)  
scaler = StandardScaler()  
df[SEdata] = scaler.fit_transform(df[SEdata])  
df = df.dropna(subset=df.columns.difference(['Country_count']))
num_rows = df.shape[0]  
print("Number of rows:", num_rows)  


In [ ]:
# Train through 2019; define future rows.
y = 'Country_count'
x = df[['Conflict'] + SEdata + label] 
mask_train = df['year'] <= 2019  
x_train = x[mask_train]  
y_train = df[y][mask_train]
mask_test = df['year'] > 2019
x_test = x[mask_test]  
y_test = df[y][mask_test]


In [ ]:
# Historical-model importance; error bars show tree dispersion.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd  
from sklearn.ensemble import RandomForestRegressor  
rfr = RandomForestRegressor(random_state=42)  
rfr.fit(x_train, y_train)  
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 8
importances = rfr.feature_importances_
feature_importances = dict(zip(list(x_train.columns), importances))   
sorted_feature_importances = sorted(feature_importances.items(), key=lambda item: item[1], reverse=True)
print("\nFeature names and importance:")
for feature, importance in sorted_feature_importances:
    print(f"{feature}: {importance:.4f}")
std = np.std([tree.feature_importances_ for tree in rfr.estimators_], axis=0)
sedata_length = len(SEdata)
label_indices = np.arange(sedata_length + 1, sedata_length + 1 + len(label))
label_importances = importances[label_indices]
label_importances_std = std[label_indices]
sorted_indices = np.argsort(label_importances)[::-1]  
sorted_label_importances = label_importances[sorted_indices]  
sorted_label_importances_std = label_importances_std[sorted_indices]  
sorted_labels = [label[i] for i in sorted_indices]  
color = "#72B3C5"
#color = "#438EBD" 
plt.figure(figsize=(4.3, 6))  
plt.title("Feature Importances")  
plt.bar(range(len(sorted_labels)), sorted_label_importances, color=color, align="center", 
        yerr=sorted_label_importances_std, capsize=5, width=0.8)
ax = plt.gca()
ax.margins(y=0.02)
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(1.0)
plt.tick_params(axis='both', which='major', length=6, width=1.2)
plt.xticks(range(len(sorted_labels)), sorted_labels, rotation=30)  
plt.xlim([-1, len(sorted_labels)])  
plt.tight_layout()  
plt.savefig(str(FIGURE3_OUTPUT_DIR / 'importance_Feature Importances3.png'), format='png', dpi=300, bbox_inches='tight')


In [ ]:
# All-feature importance table.
import pandas as pd

all_features = ['Conflict'] + SEdata + label
importances_all = importances

importance_sum = importances_all.sum()
importance_percent = importances_all / importance_sum * 100

feature_type = [
    'Policy' if f in label else 'Other'
    for f in all_features
]

df_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances_all,
    'Percentage (%)': importance_percent,
    'Type': feature_type
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)

df_importance['Importance'] = df_importance['Importance'].round(4)
df_importance['Percentage (%)'] = df_importance['Percentage (%)'].round(2)

print(df_importance)

# df_importance.to_excel(r"C:\Users\Lenovo\Desktop\feature_importance.xlsx", index=False)


In [ ]:
# SE/ENV/CC importance totals.
SE = {'population_density', 'gdp_percapita', 'rural_population', 'export'}
ENV = {'precipitation', 'bws_score'}
CC = ['Conflict'] + label

import pandas as pd

all_features = ['Conflict'] + SEdata + label
importances_all = importances

importance_sum = importances_all.sum()
importance_percent = importances_all / importance_sum * 100

def get_feature_type(f):
    if f in SE:
        return 'SE'
    elif f in ENV:
        return 'ENV'
    elif f in CC:
        return 'CC'
    else:
        return 'Other'

feature_type = [get_feature_type(f) for f in all_features]

df_importance = pd.DataFrame({
    'Feature': all_features,
    'Importance': importances_all,
    'Percentage (%)': importance_percent,
    'Type': feature_type
})

df_importance = df_importance.sort_values(by='Importance', ascending=False)

# ==============================
# ==============================
df_grouped = df_importance.groupby('Type')['Percentage (%)'].sum().reset_index()
df_grouped = df_grouped.sort_values(by='Percentage (%)', ascending=False)

df_importance['Percentage (%)'] = df_importance['Percentage (%)'].round(6)
df_importance['Importance'] = df_importance['Importance'].round(6)

df_grouped['Percentage (%)'] = df_grouped['Percentage (%)'].round(6)

# ==============================
# ==============================
print("===== Individual feature importance =====")
print(df_importance)

print("\n===== Importance percentages by category=====")
print(df_grouped)

print("\nTotal category percentage:", df_grouped['Percentage (%)'].sum().round(6))


In [ ]:
# Radial plot: transformed angular widths and rank-based radius.
import numpy as np
import matplotlib.pyplot as plt

# =========================
# =========================
def ensure_list(x):
    if isinstance(x, str):
        return x.replace(',', ' ').split()
    return x

SEdata = ensure_list(SEdata)
label = ensure_list(label)

# =========================
# =========================
all_features = list(x_train.columns)
all_features = [f.strip() for f in all_features]

values = np.array(importances)
errors = np.array(std)

assert len(all_features) == len(values)

# =========================
# =========================
sorted_idx = np.argsort(values)[::-1]

all_features = [all_features[i] for i in sorted_idx]
values = values[sorted_idx]

# =========================
# =========================
SE = {'population_density', 'gdp_percapita', 'rural_population', 'export'}
ENV = {'precipitation', 'bws_score'}
GOV = set(label)

color_map = []
for f in all_features:
    if f in SE:
        color_map.append("#007878")
    elif f in ENV:
        color_map.append("#97D5CB")
    elif f in GOV:
        color_map.append("#F5DF8B")
    else:
        color_map.append("#F5DF8B")

# =========================
# =========================
inner_colors = []
for i in range(len(values)):
    if i % 2 == 0:
        inner_colors.append("#EBEBED")
    else:
        inner_colors.append("white")

# =========================
# =========================
alpha = 0.6
values_safe = np.maximum(values, 1e-6) ** alpha

widths = values_safe / values_safe.sum() * 2 * np.pi
angles = np.cumsum(widths) - widths / 2

# =========================
# =========================
base_radius = 1.5
step = 0.10

idx = np.arange(len(values))
total_radius = base_radius * (1 + step * idx)

# =========================
# =========================
outer_heights = np.full_like(total_radius, base_radius * 0.66)
inner_heights = total_radius - outer_heights

# =========================
# =========================
fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw=dict(polar=True))

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# =========================
# =========================
ax.bar(
    angles,
    inner_heights,
    width=widths,
    bottom=0,
    color=inner_colors,
    edgecolor="white",
    linewidth=1.5
)

# =========================
# =========================
ax.bar(
    angles,
    outer_heights,
    width=widths,
    bottom=inner_heights,
    color=color_map,
    edgecolor="white",
    linewidth=1.5
)

# =========================
# =========================
label_radius = np.max(total_radius) * 1.65

for angle, feature_label in zip(angles, all_features):
    ax.text(
        angle,
        label_radius,
        feature_label,
        ha='center',
        va='center',
        fontsize=7
    )

# =========================
# =========================
ax.spines['polar'].set_visible(False)
ax.grid(False)
ax.set_yticklabels([])
ax.tick_params(axis='x', length=0)
ax.set_xticks([])

plt.tight_layout()

#plt.savefig(
#    str(FIGURE3_OUTPUT_DIR / 'importance_Nightingale.png'),
#    dpi=300,
#    bbox_inches='tight'
#)

plt.show()


In [ ]:
# Category legend with fixed percentages.
import matplotlib.pyplot as plt

categories = ['Development', 'Climate', 'Event']
values = [66.548024, 17.549831, 15.902145]
colors = ['#579F9B', '#97D5CB', '#F5DF8B']

values_ratio = [v / 100 for v in values]

plt.figure(figsize=(6, 4), dpi=150)

plt.barh(categories, values_ratio, color=colors, height=0.75)

plt.xlim(0, 0.7)

plt.xticks([])

ax = plt.gca()
ax.set_yticklabels([])

ax.tick_params(
    axis='y',
    which='both',
    length=12,
    width=2.2,
    direction='in'
)

ax.tick_params(axis='x', length=0)

#for i, v in enumerate(values):
#    plt.text(values_ratio[i] + 0.01, i, f"{v:.1f}%", va='center')

plt.gca().invert_yaxis()

ax.spines['left'].set_linewidth(3.2)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

plt.tight_layout()

plt.savefig(str(FIGURE3_OUTPUT_DIR / 'importance_Nightingale_legend.png'), dpi=300)

plt.show()
